# Module 3 — Embeddings & recherche sémantique

## Objectif
Transformer du texte en **vecteurs** et retrouver les passages **par le sens** — avec **Qdrant** (base vectorielle locale) et le générateur d'embeddings **moderne** de Microsoft.Extensions.AI.

> Prérequis : `docker compose up -d` (Ollama + Qdrant sur localhost:6334).

### Analogie
Un **embedding** = les **coordonnées GPS** d'un texte. Deux textes de sens proche ont des coordonnées proches. Qdrant = la **carte** où l'on cherche les voisins.

### Note (API moderne)
On utilise `IEmbeddingGenerator<string, Embedding<float>>` (et non l'ancien `ITextEmbeddingGenerationService`, déprécié).

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.74.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.Ollama, 1.74.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.Qdrant, 1.74.0-preview"
#pragma warning disable SKEXP0070, SKEXP0001
using Microsoft.Extensions.AI;            // IEmbeddingGenerator (moderne)
using Microsoft.Extensions.VectorData;    // attributs VectorStore + collection
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.Qdrant;
using Qdrant.Client;
using System.Security.Cryptography; using System.Text;
var endpoint = new Uri("http://localhost:11434");
// Embeddings MODERNES : IEmbeddingGenerator (remplace ITextEmbeddingGenerationService)
var embed = Kernel.CreateBuilder().AddOllamaEmbeddingGenerator("nomic-embed-text", endpoint)
    .Build().GetRequiredService<IEmbeddingGenerator<string, Embedding<float>>>();
var store = new QdrantVectorStore(new QdrantClient("localhost"), ownsClient: true);
ulong Id(string s) => BitConverter.ToUInt64(MD5.HashData(Encoding.UTF8.GetBytes(s)), 0);

In [ ]:
// Un enregistrement Qdrant : clé + texte + vecteur (768 = dimension de nomic-embed-text)
public sealed class Doc {
    [VectorStoreKey] public ulong Id { get; set; }
    [VectorStoreData] public string Text { get; set; } = "";
    [VectorStoreData(IsIndexed = true)] public string Source { get; set; } = "";
    [VectorStoreVector(768, DistanceFunction = DistanceFunction.CosineSimilarity)]
    public ReadOnlyMemory<float> Vector { get; set; }
}

In [ ]:
var collection = store.GetCollection<ulong, Doc>("demo_kb");
await collection.EnsureCollectionExistsAsync();

async Task Add(string text, string source) {
    var v = (await embed.GenerateAsync(text)).Vector;      // .Vector = ReadOnlyMemory<float>
    await collection.UpsertAsync(new Doc { Id = Id(text), Text = text, Source = source, Vector = v });
}
await Add("Wi-Fi instable : redemarrer le point d'acces.", "reseau");
await Add("Mot de passe oublie : self-service de reset.", "compte");
await Add("Application qui plante : vider le cache.", "logiciel");
Console.WriteLine("Indexé dans Qdrant.");

In [ ]:
// La question ne contient AUCUN mot-clé exact -> pourtant la bonne procédure remonte
var qv = (await embed.GenerateAsync("mon internet sans fil rame")).Vector;
await foreach (var r in collection.SearchAsync(qv, top: 2))
    Console.WriteLine($"[{r.Score:0.000}] {r.Record.Text}");

**À observer** : « internet sans fil » retrouve « Wi-Fi / point d'accès » : c'est la recherche **par le sens**.

## Récap
- Embeddings modernes (`IEmbeddingGenerator`) + Qdrant (`VectorStore`) : aucune API dépréciée.
- On a la brique « Récupérer ». On la branche sur le LLM → Module 4.